# 🚗 Understanding PID Control
## A Hands-On Guide for the Never-Crash Vehicle

---

### How to Use This Notebook

| Label | Who | What |
|-------|-----|------|
| 🎯 **DEMO** | Instructor runs | Watch, listen, ask questions |
| 🤝 **TOGETHER** | We work as a class | Shout out predictions! |
| ✏️ **PRACTICE** | You alone | Fill in the `...` to make it work |

---

### The Challenge

The **Never Crash** vehicle uses a distance sensor (max reading: **160 cm**) to
approach walls and other vehicles without colliding.

The goal: get as close as possible to the obstacle, holding at exactly **10 cm**.

- Far away (sensor reads ~160 cm)? → Full throttle toward the obstacle
- Getting close (error shrinking)? → Ease off the gas
- At 10 cm? → Hold position (vehicle following) or prepare to turn (wall)
- Too close? → Back up

Simple to say. Hard to do. By the end of this notebook you'll understand exactly
why we need PID — and what each of P, I, and D contributes.

---

### The Formula We're Building

```
throttle = Kp × error  +  Ki × ∫error dt  +  Kd × (Δerror/Δt)
            ────────────    ──────────────      ────────────────
               P term           I term               D term
          "React to NOW"   "Remember PAST"    "Predict FUTURE"
```

where **error = current_distance − target_distance**

Let's build this up one piece at a time.

In [ ]:
import numpy as np
from pid_plots import (
    plot_naive_strategies, plot_error_signal, plot_kp_comparison,
    plot_kp_tune, plot_steady_state_error, plot_integral_detail,
    plot_pi_comparison, plot_derivative_concept, plot_pd_comparison,
    plot_kd_explore, plot_pid_contributions, plot_grand_comparison,
    plot_pid_tune,
)

MAX_SENSOR = 160.0     # cm — maximum distance the sensor can read
TARGET     = 10.0      # cm — hold this distance from the wall / vehicle ahead

print('Setup complete.')
print(f'  MAX_SENSOR = {MAX_SENSOR} cm  |  TARGET = {TARGET} cm')
print(f'  Max error  = {MAX_SENSOR - TARGET:.0f} cm  (when sensor reads maximum range)')
print(f'  Kp target  ≈ {1/(MAX_SENSOR - TARGET):.4f}  (so Kp × max_error ≈ 1.0 = full throttle)')

## The Simulation Engine

The cell below is a **physics simulation** of your car and wall.
We'll use it throughout the notebook so you can *see* what PID does
before you put it on the physical vehicle.

The physics model:
- Car has mass (inertia) — it takes time to speed up or slow down
- Throttle creates force toward the wall; drag opposes motion
- `disturbance` = constant force *away* from wall (like driving up a slope)

You don't need to understand every line — just run it once, then use `simulate(...)`.


In [ ]:
def simulate(Kp=0.0, Ki=0.0, Kd=0.0,
             initial_distance=30.0, target=TARGET,
             dt=0.05, steps=300, disturbance=0.0):
    '''
    Simulate a car maintaining distance from a wall using PID.

    Positive throttle  → car moves toward wall → distance decreases
    disturbance > 0    → constant force pushing car AWAY from wall
    '''
    MAX_ACCEL = 20.0   # cm/s² at full throttle
    DRAG      = 2.0    # velocity damping (friction)

    distances  = [initial_distance]
    errors, P_log, I_log, D_log, throttle_log = [], [], [], [], []

    distance   = initial_distance
    velocity   = 0.0
    integral   = 0.0
    last_error = 0.0

    for step in range(steps):
        error      = distance - target
        integral  += error * dt
        derivative = (error - last_error) / dt if step > 0 else 0.0

        P        = Kp * error
        I        = Ki * integral
        D        = Kd * derivative
        throttle = max(-1.0, min(1.0, P + I + D))

        # Physics
        net_force = throttle * MAX_ACCEL - DRAG * velocity - disturbance
        velocity += net_force * dt
        distance  = max(0.5, distance - velocity * dt)

        errors.append(error); P_log.append(P); I_log.append(I)
        D_log.append(D);      throttle_log.append(throttle)
        distances.append(distance)
        last_error = error

    t = np.linspace(0, steps * dt, steps)
    return dict(time=t,
                distance=np.array(distances[:steps]),
                error=np.array(errors),
                P=np.array(P_log), I=np.array(I_log), D=np.array(D_log),
                throttle=np.array(throttle_log))

print('Simulation engine loaded ✓')
print('Usage: data = simulate(Kp=0.08, Ki=0.015, Kd=0.5)')

---
## Part 1 — Why Simple Control Fails

Before we learn PID, let's see what happens with *simpler* strategies.
Two common approaches:

1. **Open loop** — set throttle to a constant, ignore the sensor
2. **Bang-bang** — "if too far, go full speed; if too close, full reverse"

🎯 **DEMO** — Run the cell below and we'll discuss what goes wrong with each.


In [ ]:
# 🎯 DEMO: Two naive strategies — and why they fail

plot_naive_strategies(TARGET)

print()
print('Open loop:  ignores the sensor — crashes at any speed!')
print('Bang-bang:  sensor IS used, but overcorrects — oscillates forever')
print()
print('What we need: a response that is PROPORTIONAL to how wrong we are.')

---
## Part 2 — The Error Signal

Every PID controller is built on one idea:

> **error = current_distance − target_distance**

- `error > 0` → adjust throttle based on how close to target
- `error < 0` → car is too close → need to back up (negative throttle)

This single number is the *language* the controller speaks.
Let's visualize it.


In [ ]:
# 🎯 DEMO: The error signal

data = simulate(Kp=0.08, steps=200)
plot_error_signal(data, TARGET)

print('KEY: error = current_distance - TARGET_DISTANCE')
print('  Positive → too far   → go forward')
print('  Negative → too close → back up')
print('  Zero     → perfect!')

---
## Part 3 — Proportional Control (P)

The simplest idea: **make the throttle proportional to the error**.

```python
error    = current_distance - TARGET_DISTANCE
P        = Kp * error
throttle = P
```

- Big error → big throttle (move fast when far away)
- Small error → small throttle (ease in near the target)
- Zero error → zero throttle (stop exactly at target... in theory)

`Kp` is a **gain** — a dial that controls how aggressively the car reacts.

> **Never Crash calibration:**  At maximum sensor range, `error = 160 − 10 = 150 cm`.
> We want full throttle at that point: `Kp × 150 ≈ 1.0`, so **Kp ≈ 0.007** on the real
> vehicle.  The simulation below uses shorter distances (30 cm range) to keep the
> curves readable — the same tuning logic applies at any scale.

🎯 **DEMO** — Run the cell to see how Kp affects behavior.

In [ ]:
# 🎯 DEMO: Three different Kp values

configs = [
    (0.02, '#90CAF9', 'Kp = 0.02  (Too Small)\nSluggish — barely reacts'),
    (0.08, '#4CAF50', 'Kp = 0.08  (Good)\nDecent response'),
    (0.28, '#F44336', 'Kp = 0.28  (Too Large)\nOscillates — overcorrects'),
]
datasets = [(simulate(Kp=kp, steps=300), color, title)
            for kp, color, title in configs]
plot_kp_comparison(datasets, TARGET)

print()
print('Observation:')
print('  Small Kp → slow approach, may never quite reach target')
print('  Large Kp → fast but oscillates, overshoots, may be unstable')
print('  "Just right" → depends on the physical system!')

### 🤝 TOGETHER — Find the Best Kp

Change `KP_VALUE` below and run the cell.
Before you run: **predict** what will happen. Will it be faster? Slower? More oscillation?


In [ ]:
# 🤝 TOGETHER: Tune Kp — change this and predict before running!

KP_VALUE = 0.08   # ← Try values like 0.01, 0.05, 0.15, 0.30, 0.50

data = simulate(Kp=KP_VALUE, steps=300)
plot_kp_tune(data, KP_VALUE, TARGET)

### ✏️ PRACTICE — Implement Proportional Control

Fill in the two `...` lines below.
This is exactly the code that goes in **vehicle-2.py**.


In [ ]:
# ✏️ PRACTICE: Implement P

def p_controller(current_distance, target, Kp):
    # Step 1: calculate the error
    error = ...                          # YOUR CODE

    # Step 2: proportional term
    P = ...                              # YOUR CODE

    # Step 3: clamp to motor limits
    throttle = max(-1.0, min(P, 1.0))
    return throttle

# ── Test it ──────────────────────────────────────────────────────────────────
print('Testing your P controller:')
dist_seq = [30, 25, 20, 16, 13, 11, 10.5, 10.2, 10.05]
kp = 0.08
tgt = 10
for dist in dist_seq:
    try:
        result = p_controller(dist, tgt, kp)
        ok = (result > 0 and dist > tgt) or (result < 0 and dist < tgt) or (result == 0 and dist == tgt)
        sign = 'pos ✓' if result > 0 else ('neg ✓' if result < 0 else 'zero ✓')
        print(f'  dist={dist:5.1f}, target={tgt} → throttle={result:+.4f}  ({sign if ok else "WRONG ✗"})')
    except Exception as e:
        print(f'  Error: {e}')

---
## Part 4 — The Problem with P Alone

P control has **two failure modes**:

### 1. Oscillation (Kp too high)
Car overshoots the target, then overcorrects, then overshoots again...

### 2. Steady-State Error (real-world disturbances)
Some constant force is always pushing the car away from the target
(friction, a slope, motor imbalance, etc.).

With P-only, the car can *only* apply throttle when there's an error.
So it reaches an equilibrium **slightly off-target** where:
> `Kp × error_ss = disturbance_force`

The car will never reach exactly 10 cm — it settles at 10 + error_ss.

> **When does this matter for Never Crash?**
> - **Following a moving vehicle:** you can reach steady state — the integral
>   helps you lock onto exactly 10 cm even with motor friction pushing you back.
> - **Approaching a wall:** you probably won't hit steady state before turning,
>   but the integral still helps you slow down precisely near the target.

🎯 **DEMO** — Let's see this steady-state error in action.

In [ ]:
# 🎯 DEMO: Steady-state error — P alone can't fight a constant disturbance

data_p = simulate(Kp=0.08, disturbance=3.0, steps=400)
plot_steady_state_error(data_p, TARGET, kp=0.08, disturbance=3.0)

---
## Part 5 — Integral: The Memory of Past Mistakes

> **"How long have I been off-target?"**

The integral accumulates error over time:

```python
integral += error * dt        # add a tiny slice of error each timestep
I = Ki * integral             # integral term
```

**The key insight:** Even a tiny, persistent error adds up.

- At each timestep, you add a small slice: `error × dt`
- This is literally the **area under the error curve**
- After many timesteps, the integral becomes large enough to *overcome* the disturbance

Once the car reaches the target (error = 0), the integral **holds** its value —
maintaining exactly the throttle needed to fight the disturbance. This is called
**integral memory**.

🎯 **DEMO** — Watch the area accumulate below.


In [ ]:
# 🎯 DEMO: The Integral — area under the error curve

data = simulate(Kp=0.08, disturbance=3.0, steps=400)
plot_integral_detail(data, TARGET)

In [ ]:
# 🎯 DEMO: PI eliminates steady-state error

data_p  = simulate(Kp=0.08, Ki=0.000, disturbance=3.0, steps=400)
data_pi = simulate(Kp=0.08, Ki=0.015, disturbance=3.0, steps=400)
plot_pi_comparison(data_p, data_pi, TARGET)

### ✏️ PRACTICE — Implement the Integral Term

Three lines to fill in this time. Think about the formula:

```
integral += error * dt        (accumulate a slice each timestep)
I = Ki * integral             (take a fraction of the total)
```


In [ ]:
# ✏️ PRACTICE: Implement the Integral

def pi_controller(current_distance, target, Kp, Ki,
                  integral, dt):
    '''
    Returns (throttle, updated_integral).
    integral and dt are passed in from the calling loop.
    '''
    # Step 1: error
    error = current_distance - target

    # Step 2: Proportional
    P = Kp * error

    # Step 3: Integral — accumulate the error over time
    integral += ...            # YOUR CODE  (hint: error × dt)
    I = ...                    # YOUR CODE  (hint: Ki × integral)

    throttle = max(-1.0, min(P + I, 1.0))
    return throttle, integral

# ── Test ──────────────────────────────────────────────────────────────────────
print('Testing Integral implementation:')
integ = 0.0
dist_seq = [30, 25, 20, 16, 13, 11, 10.5, 10.2, 10.05]
for d in dist_seq:
    try:
        thr, integ = pi_controller(d, TARGET, 0.08, 0.015, integ, 0.05)
        print(f'  dist={d:5.2f}  integral={integ:7.3f}  throttle={thr:+.4f}')
    except Exception as e:
        print(f'  Error: {e}')
        break

---
## Part 6 — Derivative: Anticipating the Future

> **"How fast is the error changing?"**

The derivative measures the *rate of change* of the error — the slope of the error curve:

```python
derivative = (error - last_error) / dt
D = Kd * derivative
```

**The key insight:** If the car is approaching fast, the error is dropping quickly
(large negative derivative). The D term adds a *braking* force to prevent overshoot.

- Steep negative slope → approaching quickly → D reduces throttle (slow down!)
- Steep positive slope → moving away quickly → D increases throttle (resist it!)
- Flat slope → error not changing → D ≈ 0

This is why D is called the **predictive** term — it reacts to *how things are changing*,
not just what the error is right now.

🎯 **DEMO** — Let's see the derivative as slope on the error curve.


In [ ]:
# 🎯 DEMO: Derivative = slope of the error curve

plot_derivative_concept()

print()
print('Large NEGATIVE derivative → car is closing in fast → D REDUCES throttle → prevents crash')
print('Large POSITIVE derivative → car is moving away fast → D INCREASES throttle → fights it')
print('Near-zero derivative      → error not changing much → D contributes little')

In [ ]:
# 🎯 DEMO: D reduces oscillation

data_p  = simulate(Kp=0.25, Kd=0.0, steps=300)
data_pd = simulate(Kp=0.25, Kd=1.2, steps=300)
plot_pd_comparison(data_p, data_pd, TARGET)

print()
print('Why does D damp oscillation?')
print('  As car zooms past target, error changes fast (big negative derivative)')
print('  D term fires a strong REVERSE signal → slows the car before it overshoots')
print('  Think of D as: the faster you are heading somewhere wrong, the harder you brake')

### 🤝 TOGETHER — Explore the Derivative

Change `KD_VALUE` below. Try values between 0 and 3.0.
**Predict first:** What happens if Kd is very large?


In [ ]:
# 🤝 TOGETHER: Explore Kd (start with Kp=0.20 so oscillation is visible)

KD_VALUE = 0.0   # ← Try 0, 0.3, 0.8, 1.5, 3.0, 6.0

data = simulate(Kp=0.20, Kd=KD_VALUE, steps=300)
plot_kd_explore(data, KD_VALUE, kp_value=0.20, target=TARGET)

### ✏️ PRACTICE — Implement the Derivative Term

```
derivative = (current_error - last_error) / dt
D = Kd * derivative
```

`last_error` is what the error was in the *previous loop iteration*. That's why
vehicle-2.py keeps a global variable `last_error` — to remember it between calls.


In [ ]:
# ✏️ PRACTICE: Implement the Derivative

def pid_controller(current_distance, target, Kp, Ki, Kd,
                   last_error, integral, dt):
    '''
    Full PID. Returns (throttle, error, integral).
    Caller stores error as last_error for the next call.
    '''
    error = current_distance - target
    P     = Kp * error

    integral += error * dt
    I         = Ki * integral

    # ── YOUR CODE: derivative and D term ──────────────────────────────────────
    derivative = ...       # (current error - last error) / dt
    D          = ...       # Kd × derivative

    throttle = max(-1.0, min(P + I + D, 1.0))
    return throttle, error, integral

# ── Test ──────────────────────────────────────────────────────────────────────
print('Testing full PID controller:')
last_err = 0.0
integ    = 0.0
for dist in [30, 24, 18, 14, 12, 10.8, 10.2, 10.05]:
    try:
        thr, last_err, integ = pid_controller(
            dist, TARGET, 0.08, 0.015, 0.5, last_err, integ, 0.05)
        print(f'  dist={dist:5.2f}  P+I+D → throttle={thr:+.4f}')
    except Exception as e:
        print(f'  Error: {e}'); break

---
## Part 7 — Full PID in Action

Now we combine all three terms:

```
throttle = Kp × error  +  Ki × integral  +  Kd × derivative
```

Each term handles a different failure mode:

| Term | Question it answers | Fixes |
|------|---------------------|-------|
| **P** | "How wrong am I *right now*?" | No response at all |
| **I** | "How long have I *been* wrong?" | Steady-state error |
| **D** | "How *fast* is the error changing?" | Oscillation / overshoot |

🎯 **DEMO** — First, let's see each term's contribution to the throttle signal.
Then the grand comparison: P vs PI vs PD vs PID.


In [ ]:
# 🎯 DEMO: How P, I, and D each contribute to the throttle signal

data = simulate(Kp=0.08, Ki=0.015, Kd=0.5, disturbance=3.0, steps=400)
plot_pid_contributions(data, TARGET)

print()
print('Notice:')
print('  P  — large at start, shrinks as car approaches, settles near zero')
print('  I  — starts at zero, grows slowly, eventually overcomes the disturbance')
print('  D  — spikes on fast changes, damps oscillation, nearly zero once settled')

In [ ]:
# 🎯 DEMO: The Grand Comparison — P vs PI vs PD vs PID (all with disturbance)

configs = [
    ('P only', 0.08, 0.000, 0.0, '#FF8C00', "Has steady-state error\n(can't fight the disturbance)"),
    ('PI',     0.08, 0.015, 0.0, '#2196F3', 'Fixes steady-state error\nbut may be slower or oscillate'),
    ('PD',     0.08, 0.000, 0.5, '#9C27B0', 'Reduces oscillation\nbut still has steady-state error'),
    ('PID',    0.08, 0.015, 0.5, '#2E7D32', 'Best of everything:\nfast, stable, no steady-state error'),
]
datasets = [
    (simulate(Kp=kp, Ki=ki, Kd=kd, disturbance=3.0, steps=400),
     label, kp, ki, kd, color, desc)
    for label, kp, ki, kd, color, desc in configs
]
plot_grand_comparison(datasets, TARGET)

### 🤝 TOGETHER — Tune Your Own PID

Now you control all three gains. The scoring function will tell you how well you did.

**Goal:** reach the target as fast as possible, with as little overshoot as possible,
and hold it there with minimal steady-state error.


In [ ]:
# 🤝 TOGETHER: Tune your own PID — adjust all three gains

KP = 0.08     # ← Tune this
KI = 0.015    # ← Tune this
KD = 0.5      # ← Tune this

data = simulate(Kp=KP, Ki=KI, Kd=KD, disturbance=3.0, steps=400)
plot_pid_tune(data, KP, KI, KD, TARGET)

---
## Part 8 — Connecting to Your Vehicle

Everything you've learned maps directly to the `get_pid_throttle()` function
in **vehicle-2.py**. Here's the bridge:

| Notebook | vehicle-2.py | What it is |
|----------|-------------|-----------|
| `TARGET` | `TARGET_DISTANCE = 10` | 10 cm goal |
| `MAX_SENSOR` | *(sensor limit)* | 160 cm max reading |
| `Kp` | `KP ≈ 0.007` | Proportional gain |
| `Ki` | `KI = 0.0001` | Integral gain |
| `Kd` | `KD = 0.002` | Derivative gain |
| `integral += error * dt` | `integral += error * dt` | Accumulate error |
| `derivative = Δerror/Δt` | `(error - last_error) / dt` | Rate of change |
| `P + I + D` | `P + I + D` → `speed` | Total output |

### Kp Calibration for the 160 cm Sensor

The sensor maxes out at **160 cm**.  At that reading:

```
error_max = MAX_SENSOR − TARGET = 160 − 10 = 150 cm
```

We want full throttle (`= 1.0`) at maximum range, so:

```
Kp × 150 ≈ 1.0   →   Kp ≈ 0.0067
```

This is the starting point for proportional gain on the real vehicle.
Adjust up slightly if the approach feels sluggish; adjust down if it
oscillates near the target.

### Steady-State vs. Wall Approach

| Scenario | Steady state? | Notes |
|----------|--------------|-------|
| Following another vehicle | Yes — both move together | I term locks in exactly 10 cm |
| Approaching a stationary wall | Unlikely — you'll turn first | I term still improves precision |

The vehicle runs the PID loop at **20 Hz** (every 0.05 s), exactly like our simulation.

✏️ **PRACTICE** — Complete the full `get_pid_throttle()` function below.
This is your blueprint for what to fill into vehicle-2.py.

In [ ]:
# ✏️ PRACTICE: Complete the full PID function — mirrors vehicle-2.py exactly

import time as _time

KP              = 0.08      # proportional gain
KI              = 0.015     # integral gain
KD              = 0.5       # derivative gain
TARGET_DISTANCE = 10.0      # cm
MAX_SPEED       = 1.0
MIN_SPEED       = 0.0

# --- tracking variables (mirrors vehicle-2.py globals) ---
last_error = 0.0
integral   = 0.0
last_time  = _time.monotonic()

def get_pid_throttle(current_dist):
    global last_error, integral, last_time

    now = _time.monotonic()
    dt  = now - last_time
    if dt <= 0:
        dt = 0.001

    # ── 1. Error ──────────────────────────────────────────────────────────────
    error = ...                   # YOUR CODE: current_dist - TARGET_DISTANCE

    # ── 2. Proportional ───────────────────────────────────────────────────────
    P = ...                       # YOUR CODE: KP × error

    # ── 3. Integral ───────────────────────────────────────────────────────────
    integral += ...               # YOUR CODE: accumulate error × dt
    I = ...                       # YOUR CODE: KI × integral

    # ── 4. Derivative ─────────────────────────────────────────────────────────
    derivative = ...              # YOUR CODE: (error - last_error) / dt
    D = ...                       # YOUR CODE: KD × derivative

    # ── 5. Save state for next call ───────────────────────────────────────────
    last_error = ...              # YOUR CODE
    last_time  = ...              # YOUR CODE

    # ── 6. Combine and clamp ──────────────────────────────────────────────────
    speed = P + I + D
    return max(MIN_SPEED, min(speed, MAX_SPEED))


# ── Test your implementation ──────────────────────────────────────────────────
print('Testing your get_pid_throttle():')
last_error = 0.0; integral = 0.0; last_time = _time.monotonic()

test_distances = [30, 24, 18, 14, 12, 11, 10.5, 10.2, 10.05, 10.01]
for dist in test_distances:
    try:
        thr = get_pid_throttle(dist)
        direction = 'forward' if thr > 0.01 else ('stop' if thr < 0.01 else 'stop')
        print(f'  dist={dist:5.2f} cm → throttle={thr:+.4f}  ({direction})')
    except Exception as e:
        print(f'  Error at dist={dist}: {e}')
        break

print()
print('If these all returned sensible values → you are ready for vehicle-2.py!')

### Tuning Intuition for Never Crash

When you run your vehicle, it probably won't behave exactly like the simulation —
that's normal. Use these heuristics:

| Symptom | Likely cause | Try |
|---------|-------------|-----|
| Vehicle crawls toward target, never quite arrives | Kp too small | Increase Kp |
| Vehicle oscillates / bounces around the target | Kp too large | Decrease Kp |
| Reaches close but always stops a bit short (with friction) | Ki = 0 or too small | Increase Ki |
| Adding Ki causes slow, growing oscillation | Ki too large | Decrease Ki |
| Overshoots badly at high Kp | Kd = 0 or too small | Increase Kd |
| Jerks / chatters nervously | Kd too large (amplifying sensor noise) | Decrease Kd |

**Starting point for Never Crash:** `Kp ≈ 0.007`, `Ki ≈ 0.0001`, `Kd ≈ 0.002`

**The golden rule:** Tune P first, then add I, then add D.

---

## Summary

```
throttle = Kp × error              ← P: react to current error
         + Ki × integral           ← I: eliminate persistent drift
         + Kd × derivative         ← D: damp oscillation, prevent overshoot
```

P answers: **"What's wrong right now?"**
I answers: **"How long has it been wrong?"**
D answers: **"How fast is it changing?"**

Together, they give the Never Crash vehicle smooth, precise, self-correcting motion.

**Now go make your vehicle never crash. 🚗🧱**